# GaussLaw — 2D and 3D projection of a non-solenoidal magnetic field

`MagneticStandalone` · opt-in Gauss-law correction.

This merged form runs both the 2D and 3D projections from one shared data template. The 2D case uses two magnetic components and the MPI path; the 3D extrusion uses the full three-component vector and remains sequential.


In [ ]:
from trustutils import run
run.TRUST_parameters("1.9.8")
run.reset()


## Run both dimensions

The 2D ladder runs on two MPI ranks. The 3D ladder is sequential, uses the full-vector extrusion, and converts its Pave mesh with `Polyedriser dom`.


In [ ]:
import os
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

from string import Template
from trustutils import run

NB_PROCS = 2
run.reset()

# Mesh geometries and physics/post-processing blocks are the two dimensions
# of the same src template. The 3D case uses a distinct target data name
# and build subtree, so partition zones and probe files cannot collide.
MESH_BLOCKS = {
    2: r"""Mailler dom
{
    Pave pavé
    {
        Origine              0.  0.
        Nombre_de_Noeuds     $nx $ny
        Longueurs            1.  1.
    }
    {
        Bord gauche   X = 0.    0. <= Y <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.
    }
}""",
    3: r"""Mailler dom
{
    Pave pavé
    {
        Origine              0.  0.  0.
        Nombre_de_Noeuds     $nx $ny $nz
        Longueurs            1.  1.  1.
    }
    {
        Bord gauche   X = 0.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord droite   X = 1.    0. <= Y <= 1.  0. <= Z <= 1.
        Bord bas      Y = 0.    0. <= X <= 1.  0. <= Z <= 1.
        Bord haut     Y = 1.    0. <= X <= 1.  0. <= Z <= 1.
        Bord devant   Z = 0.    0. <= X <= 1.  0. <= Y <= 1.
        Bord derriere Z = 1.    0. <= X <= 1.  0. <= Y <= 1.
    }
}
Polyedriser dom""",
}
DIMENSION_BLOCKS = {
    2: {
        'boundary_block': r"""boundary_conditions
        {
            gauche  Neumann_Paroi_adiabatique
            droite  Neumann_Paroi_adiabatique
            bas     Neumann_Paroi_adiabatique
            haut    Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
  {
    Sondes
    {
      sonde_div_1 grav divergence_magnetique Periode 1e-6 Points 1 0.25 0.50
      sonde_div_2 grav divergence_magnetique Periode 1e-6 Points 1 0.50 0.50
      sonde_div_3 grav divergence_magnetique Periode 1e-6 Points 1 0.75 0.50
    }
  }""",
        'ncomp': r"""2""",
        'initial_block': r"""initial_conditions { magnetic_field Champ_Fonc_xyz dom 2 cos(2*pi*x) 0. }""",
    },
    3: {
        'boundary_block': r"""boundary_conditions
        {
            gauche   Neumann_Paroi_adiabatique
            droite   Neumann_Paroi_adiabatique
            bas      Neumann_Paroi_adiabatique
            haut     Neumann_Paroi_adiabatique
            devant   Neumann_Paroi_adiabatique
            derriere Neumann_Paroi_adiabatique
        }""",
        'post_block': r"""Postraitement
    {
        Sondes
        {
            sonde_div_1 grav divergence_magnetique Periode 1e-6 Points 1 0.25 0.50 0.50
            sonde_div_2 grav divergence_magnetique Periode 1e-6 Points 1 0.50 0.50 0.50
            sonde_div_3 grav divergence_magnetique Periode 1e-6 Points 1 0.75 0.50 0.50
        }
    }""",
        'ncomp': r"""3""",
        'initial_block': r"""initial_conditions { magnetic_field Champ_Fonc_xyz dom 3 x*(1-x) 0. 0. }""",
    },
}

MESHES_2D = {'coarse': {'nx': 17, 'ny': 17}, 'standard': {'nx': 33, 'ny': 33}, 'fine': {'nx': 65, 'ny': 65}}
MESHES_3D = {'coarse': {'nx': 17, 'ny': 4, 'nz': 4}, 'standard': {'nx': 33, 'ny': 4, 'nz': 4}, 'fine': {'nx': 65, 'ny': 4, 'nz': 4}}

def register_dimension(dim, meshes, prefix="", target_data=None, nb_procs=NB_PROCS, partition=True):
    registered = {}
    for level, mesh in meshes.items():
        dic = {"dim": dim,
               "mesh_block": Template(MESH_BLOCKS[dim]).substitute(**mesh),
               **DIMENSION_BLOCKS[dim]}
        target = f"{prefix}/{level}" if prefix else level
        kwargs = {"nbProcs": nb_procs, "excluNR": True}
        if target_data is not None:
            kwargs["targetData"] = target_data
        registered[level] = run.addCaseFromTemplate('gauss_law.data', target, dic=dic, **kwargs)
        if partition:
            registered[level].partition()
    return registered

cases = register_dimension(2, MESHES_2D)
cases_3d = register_dimension(3, MESHES_3D, prefix="3D", target_data='gauss_law_3d.data', nb_procs=1, partition=False)
case = cases["coarse"]
case_3d = cases_3d["coarse"]
run.printCases()
run.runCases()


## 2D Gauss-law checks

The 2D field starts from `b=(cos(2*pi*x), 0)`. The t=0 probe precedes the initial projection hook, so assertions use samples after the first time step.


In [ ]:
def read_probe(level, tag):
    case_name = cases[level].dataFileName_
    candidates = (f"{BUILD}/{level}/{case_name}_{tag}.son",
                  f"{BUILD}/{case_name}_{tag}.son")
    path = next((candidate for candidate in candidates if os.path.exists(candidate)), None)
    assert path is not None, f"missing probe file: {candidates}"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"empty probe file: {path}"
    t, values = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(values, dtype=float)

TAGS = ("SONDE_DIV_1", "SONDE_DIV_2", "SONDE_DIV_3")
for level in ("coarse", "standard", "fine"):
    worst = 0.0
    for tag in TAGS:
        t, values = read_probe(level, tag)
        after_initial_projection = t > 0.
        assert np.any(after_initial_projection), f"no post-initialization samples for {level}/{tag}"
        worst = max(worst, float(np.max(np.abs(values[after_initial_projection]))))
    print(f"{level:>8}: max |divergence_magnetique| = {worst:.4e}")
    assert worst < 1.e-8, f"Gauss-law correction failed on {level}: {worst:.3e}"
print("OK: interior magnetic divergence remains at solver tolerance across the mesh scan.")


## 3D Gauss-law checks

The full vector field starts from `b=(x*(1-x), 0, 0)` and has compatible net boundary flux. As in 2D, the assertions use samples after the initial projection hook.

In [ ]:
def read_probe_3d(level, tag):
    case_name = cases_3d[level].dataFileName_
    candidates = (f"{BUILD}/3D/{level}/{case_name}_{tag}.son",
                  f"{BUILD}/3D/{case_name}_{tag}.son")
    path = next((candidate for candidate in candidates if os.path.exists(candidate)), None)
    assert path is not None, f"missing probe file: {candidates}"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"empty probe file: {path}"
    t, values = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(values, dtype=float)

TAGS = ("SONDE_DIV_1", "SONDE_DIV_2", "SONDE_DIV_3")
for level in ("coarse", "standard", "fine"):
    worst = 0.0
    for tag in TAGS:
        t, values = read_probe_3d(level, tag)
        after_initial_projection = t > 0.
        assert np.any(after_initial_projection), f"no post-initialization samples for {level}/{tag}"
        worst = max(worst, float(np.max(np.abs(values[after_initial_projection]))))
    print(f"{level:>8}: max |divergence_magnetique| = {worst:.4e}")
    assert worst < 1.e-8, f"Gauss-law correction failed on {level}: {worst:.3e}"
print("OK: interior magnetic divergence remains at solver tolerance across the 3D mesh scan.")


## Scope

This projection enforces the module’s Green–Gauss divergence measure. It does not claim that the separate MPFA diffusive-flux divergence is identically zero. The unit tests additionally verify `L phi = D(G phi)`, one-pass removal of a compatible synthetic divergence, idempotence, and rejection of nonzero net boundary flux.